# NB1 — ScienceWorld Exploration and Environment Validation

**Project:** *Do AI Agents Know Which Actions Cannot Be Undone?*

**Purpose:** establish a reproducible understanding of ScienceWorld before building reversibility labels.

This notebook is exploratory. It does **not** create definitive reversibility labels, generate the final research dataset, load language models, or run hidden-state extraction.

## Research safeguards

- `done=True` is a task outcome, not proof of irreversibility.
- Matching observations or inventories are partial evidence, not proof that the complete simulator state was restored.
- Record simulator state, task progress, reward, and move count separately.
- Do not infer that an action is irreversible merely because a short recovery attempt failed.
- Keep all experiments inside the ScienceWorld simulator.
- This is a fresh rebuild. Do not import old datasets, reports, notebooks, or artifacts from the legacy project backup.

**Storage convention**

- Local working directory: `/content/irrev`
- Main Drive project: `/content/drive/MyDrive/ir`
- Independent backup snapshots: `/content/drive/MyDrive/ir_final`


## 1. Initialise the fresh project workspace

The setup cell mounts Drive, creates the agreed directory structure, and refuses to silently mix this rebuild with files already present in the local project root or the new Drive roots.

If it stops because a directory is non-empty, inspect that directory before proceeding. Do not delete or overwrite files automatically.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, time, hashlib, shutil, subprocess, platform
from collections import Counter
import importlib.metadata

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/irrev")
DRIVE_ROOT = Path("/content/drive/MyDrive/ir")
BACKUP_ROOT = Path("/content/drive/MyDrive/ir_final")

# Permit only the empty directory skeleton created by the earlier setup cell.
# Any other pre-existing files cause a stop so legacy artifacts cannot be mixed in.
if PROJECT_ROOT.exists():
    existing_files = [p for p in PROJECT_ROOT.rglob("*") if p.is_file()]
    allowed_config = PROJECT_ROOT / "project_config.json"
    unexpected_files = [p for p in existing_files if p != allowed_config]
    if unexpected_files:
        raise RuntimeError(
            "The local project directory contains files beyond the initial "
            "setup configuration. This notebook will not mix them into the "
            "fresh rebuild. Inspect/move them before proceeding: "
            + ", ".join(str(p) for p in unexpected_files[:10])
        )

for path in (DRIVE_ROOT, BACKUP_ROOT):
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(
            f"{path} already contains files. Stop and inspect it before "
            "proceeding; this notebook will not overwrite existing project data."
        )

DIRS = [
    PROJECT_ROOT / "data" / "scienceworld_exploration",
    PROJECT_ROOT / "results" / "scienceworld_exploration",
    PROJECT_ROOT / "reports",
    PROJECT_ROOT / "logs",
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "tests",
    PROJECT_ROOT / "notebooks",
    DRIVE_ROOT,
    BACKUP_ROOT,
]
for d in DIRS:
    d.mkdir(parents=True, exist_ok=True)

REPORT_DIR = PROJECT_ROOT / "reports"
DATA_DIR = PROJECT_ROOT / "data" / "scienceworld_exploration"
RESULT_DIR = PROJECT_ROOT / "results" / "scienceworld_exploration"

TASK_NAME = "boil"
VARIATION_INDEX = 0
SIMPLIFICATION = "easy"
ENV_STEP_LIMIT = 100

os.environ["JAVA_TOOL_OPTIONS"] = "-Xlog:os+container=off"

PROJECT_CONFIG = {
    "project": "Do AI Agents Know Which Actions Cannot Be Undone?",
    "notebook": "01_scienceworld_exploration.ipynb",
    "project_root": str(PROJECT_ROOT),
    "drive_root": str(DRIVE_ROOT),
    "backup_root": str(BACKUP_ROOT),
    "task_name": TASK_NAME,
    "variation_index": VARIATION_INDEX,
    "simplification": SIMPLIFICATION,
    "env_step_limit": ENV_STEP_LIMIT,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "rebuild_status": "fresh",
    "legacy_artifacts_imported": False,
}
config_path = PROJECT_ROOT / "project_config.json"
if config_path.exists():
    previous = json.loads(config_path.read_text(encoding="utf-8"))
    if previous.get("project_root") != str(PROJECT_ROOT):
        raise RuntimeError("Existing project_config.json belongs to a different project.")
else:
    config_path.write_text(json.dumps(PROJECT_CONFIG, indent=2), encoding="utf-8")

print("Project workspace initialised.")
print("Local:", PROJECT_ROOT)
print("Main Drive:", DRIVE_ROOT)
print("Backup Drive:", BACKUP_ROOT)
print("Task:", TASK_NAME, "| variation:", VARIATION_INDEX)
print("Legacy artifacts imported: False")


Mounted at /content/drive
Project workspace initialised.
Local: /content/irrev
Main Drive: /content/drive/MyDrive/ir
Backup Drive: /content/drive/MyDrive/ir_final
Task: boil | variation: 0
Legacy artifacts imported: False


## 2. Verify the runtime and install the pinned ScienceWorld version

The previous exploration used `scienceworld==1.2.3` and OpenJDK 21. This cell records the current Python/Java environment and installs the pinned package if needed.


In [2]:
print("Python:", sys.version)
print("Platform:", platform.platform())

java_result = subprocess.run(
    ["java", "-version"], capture_output=True, text=True
)
java_text = (java_result.stderr or java_result.stdout).strip()
print("\nJava version output:\n", java_text)

try:
    installed_version = importlib.metadata.version("scienceworld")
except importlib.metadata.PackageNotFoundError:
    installed_version = None

if installed_version != "1.2.3":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", "scienceworld==1.2.3"
    ])

import scienceworld
from scienceworld import ScienceWorldEnv
import inspect

SCIENCEWORLD_VERSION = importlib.metadata.version("scienceworld")
print("\nScienceWorld version:", SCIENCEWORLD_VERSION)
print("Package location:", scienceworld.__file__)
print("Constructor:", inspect.signature(ScienceWorldEnv))

if SCIENCEWORLD_VERSION != "1.2.3":
    raise RuntimeError(f"Expected ScienceWorld 1.2.3, found {SCIENCEWORLD_VERSION}")


Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39

Java version output:
 Picked up JAVA_TOOL_OPTIONS: -Xlog:os+container=off
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)

ScienceWorld version: 1.2.3
Package location: /usr/local/lib/python3.13/dist-packages/scienceworld/__init__.py
Constructor: (taskName: str = None, serverPath: str = None, envStepLimit: int = 100)


## 3. Inspect the public environment API

Capture the methods available in the installed package. This helps prevent accidental reliance on guessed method names.


In [3]:
public_methods = sorted(
    name for name in dir(ScienceWorldEnv)
    if not name.startswith("_") and callable(getattr(ScienceWorldEnv, name, None))
)

print("Public callable methods on ScienceWorldEnv:")
for name in public_methods:
    try:
        print(f"- {name}{inspect.signature(getattr(ScienceWorldEnv, name))}")
    except Exception:
        print(f"- {name}")

required_methods = [
    "load", "reset", "step", "get_task_names",
    "getObjectTree", "get_valid_action_object_combinations",
    "get_num_moves", "get_run_history", "get_task_description"
]
missing = [m for m in required_methods if not hasattr(ScienceWorldEnv, m)]
print("\nRequired API methods missing:", missing)
if missing:
    raise RuntimeError("Installed ScienceWorld API differs from the expected interface.")


Public callable methods on ScienceWorldEnv:
- clearRunHistories(self)
- clear_run_histories(self) -> None
- close(self) -> None
- getAllObjectIdsTypesReferentsLUTJSON(self)
- getAllObjectTypesLUTJSON(self)
- getGoalProgressStr(self)
- getGoldActionSequence(self)
- getMaxVariations(self, taskName)
- getNumMoves(self)
- getObjectTree(self)
- getObjectTypes(self)
- getPossibleActionObjectCombinations(self)
- getPossibleActions(self)
- getPossibleActionsWithIDs(self)
- getPossibleObjectReferentLUT(self)
- getPossibleObjectReferentTypesLUT(self)
- getPossibleObjects(self)
- getPossibleSimplifications(self)
- getRandomVariationDev(self)
- getRandomVariationTest(self)
- getRandomVariationTrain(self)
- getRunHistory(self)
- getRunHistorySize(self)
- getSimplificationsUsed(self)
- getTaskDescription(self)
- getTaskNames(self)
- getValidActionObjectCombinations(self)
- getValidActionObjectCombinationsWithTemplates(self)
- getVariationsDev(self)
- getVariationsTest(self)
- getVariationsTrain(self

## 4. Create and load the exploratory environment

ScienceWorld is initialised with the task name, then loaded with a variation and simplification. The reset result is captured without assuming more than the documented tuple structure.


In [4]:
env = ScienceWorldEnv(
    taskName=TASK_NAME,
    serverPath=None,
    envStepLimit=ENV_STEP_LIMIT
)

load_result = env.load(
    taskName=TASK_NAME,
    variationIdx=VARIATION_INDEX,
    simplificationStr=SIMPLIFICATION
)
print("Load result:", load_result)

reset_result = env.reset()
if not isinstance(reset_result, tuple) or len(reset_result) != 2:
    raise RuntimeError(
        f"Unexpected reset return structure: {type(reset_result)}"
    )

initial_observation, initial_info = reset_result
print("\nInitial observation:\n", initial_observation)
print("\nInitial info keys:", list(initial_info.keys()) if isinstance(initial_info, dict) else type(initial_info))
print("Initial move count:", env.get_num_moves())
print("\nTask description:\n", env.get_task_description())


Load result: None

Initial observation:
 This room is called the hallway. In it, you see: 
	the agent
	a substance called air
	a picture
You also see:
	A door to the art studio (that is open)
	A door to the bedroom (that is open)
	A door to the greenhouse (that is open)
	A door to the kitchen (that is open)
	A door to the living room (that is open)
	A door to the workshop (that is open)

Initial info keys: ['moves', 'score', 'reward', 'look', 'inv', 'taskDesc', 'valid', 'variationIdx', 'taskName', 'simplificationStr']
Initial move count: 0

Task description:
 Your task is to boil water. For compounds without a boiling point, combusting the substance is also acceptable. First, focus on the substance. Then, take actions that will cause it to change its state of matter.


## 5. Discover available tasks and variations

This is an inventory step only. It does not assume that all tasks or variations are equally suitable for reversibility experiments.


In [5]:
task_names = list(env.get_task_names())
task_inventory = {
    "scienceworld_version": SCIENCEWORLD_VERSION,
    "task_count": len(task_names),
    "task_names": task_names,
}
print("Available task count:", len(task_names))
for task in task_names:
    print("-", task)

(RESULT_DIR / "task_inventory.json").write_text(
    json.dumps(task_inventory, indent=2), encoding="utf-8"
)


Available task count: 30
- boil
- change-the-state-of-matter-of
- chemistry-mix
- chemistry-mix-paint-secondary-color
- chemistry-mix-paint-tertiary-color
- find-animal
- find-living-thing
- find-non-living-thing
- find-plant
- freeze
- grow-fruit
- grow-plant
- identify-life-stages-1
- identify-life-stages-2
- inclined-plane-determine-angle
- inclined-plane-friction-named-surfaces
- inclined-plane-friction-unnamed-surfaces
- lifespan-longest-lived
- lifespan-longest-lived-then-shortest-lived
- lifespan-shortest-lived
- measure-melting-point-known-substance
- measure-melting-point-unknown-substance
- melt
- mendelian-genetics-known-plant
- mendelian-genetics-unknown-plant
- power-component
- power-component-renewable-vs-nonrenewable-energy
- test-conductivity
- test-conductivity-of-unknown-substances
- use-thermometer


1034

## 6. Inspect the initial object tree

The object tree is one observable view of the environment. It may not contain every relevant simulator variable, task flag, agent variable, or action-history detail. We therefore preserve the raw tree and avoid calling it a complete state snapshot.


In [6]:
def get_object_tree(env):
    tree = env.getObjectTree()
    if isinstance(tree, str):
        try:
            return json.loads(tree)
        except json.JSONDecodeError:
            return tree
    return tree

initial_tree = get_object_tree(env)

def json_safe(value):
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)

initial_state_observation = {
    "task": TASK_NAME,
    "variation": VARIATION_INDEX,
    "observation": str(initial_observation),
    "move_count": int(env.get_num_moves()),
    "object_tree": json_safe(initial_tree),
    "task_description": str(env.get_task_description()),
}
(RESULT_DIR / "initial_environment_observation.json").write_text(
    json.dumps(initial_state_observation, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

tree_text = json.dumps(initial_tree, indent=2, ensure_ascii=False) if not isinstance(initial_tree, str) else initial_tree
print(tree_text[:12000])
print("\nObject tree type:", type(initial_tree).__name__)
print("Saved initial observation and object tree.")


{
  "uuid": "17199",
  "name": "universe",
  "type": "",
  "isDeleted": false,
  "propMaterial": null,
  "propEdibility": null,
  "propContainer": {
    "isContainer": "true",
    "isOpen": true,
    "isClosable": false
  },
  "propDevice": null,
  "propHeatSource": null,
  "propCoolingSource": null,
  "propPortal": null,
  "propMoveable": {
    "isMovable": false
  },
  "propElectricalConnection": null,
  "propLife": null,
  "contents": {
    "terminal 1-17197": {
      "uuid": "17197",
      "name": "terminal 1",
      "type": "",
      "isDeleted": false,
      "propMaterial": null,
      "propEdibility": null,
      "propContainer": null,
      "propDevice": null,
      "propHeatSource": null,
      "propCoolingSource": null,
      "propPortal": null,
      "propMoveable": {
        "isMovable": false
      },
      "propElectricalConnection": {
        "connectedTo": []
      },
      "propLife": null,
      "contents": {}
    },
    "bedroom-17325": {
      "uuid": "17325",
     

## 7. Inspect the currently valid action catalogue

`get_valid_action_object_combinations()` returns state-dependent action strings. The list is an action catalogue for the current state, not a universal list of all possible actions.


In [7]:
valid_actions_initial = env.get_valid_action_object_combinations()
if valid_actions_initial is None:
    valid_actions_initial = []

valid_actions_initial = list(valid_actions_initial)
print("Valid action count:", len(valid_actions_initial))
for action in valid_actions_initial[:200]:
    print("-", action)

(RESULT_DIR / "initial_valid_actions.json").write_text(
    json.dumps(valid_actions_initial, indent=2, ensure_ascii=False),
    encoding="utf-8"
)


Valid action count: 115
- close art studio door
- close bedroom door
- close door to greenhouse
- close door to kitchen
- close door to living room
- close door to workshop
- eat orange
- focus on agent
- focus on air
- focus on art studio
- focus on art studio door
- focus on bedroom
- focus on bedroom door
- focus on door to greenhouse
- focus on door to kitchen
- focus on door to living room
- focus on door to workshop
- focus on greenhouse
- focus on hallway
- focus on inventory
- focus on kitchen
- focus on living room
- focus on orange
- focus on picture
- focus on workshop
- go to art studio
- go to art studio door
- go to bedroom
- go to bedroom door
- go to door to greenhouse
- go to door to kitchen
- go to door to living room
- go to door to workshop
- go to greenhouse
- go to kitchen
- go to living room
- go to workshop
- inventory
- look around
- look at agent
- look at air
- look at art studio
- look at art studio door
- look at bedroom
- look at bedroom door
- look at doo

2780

## 8. Define reusable state-observation helpers

These helpers collect multiple observable channels. They are not claimed to expose the complete internal simulator state.


In [8]:
def collect_observation(env, observation=None, info=None, reward=None, done=None, action=None):
    tree = get_object_tree(env)
    return {
        "action": action,
        "observation": None if observation is None else str(observation),
        "info": json_safe(info),
        "reward": None if reward is None else float(reward),
        "done": None if done is None else bool(done),
        "move_count": int(env.get_num_moves()),
        "object_tree": json_safe(tree),
        "run_history": json_safe(env.get_run_history()),
    }

def stable_json_hash(value):
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def compact_state_summary(snapshot):
    return {
        "move_count": snapshot["move_count"],
        "observation_sha256": stable_json_hash(snapshot["observation"]),
        "object_tree_sha256": stable_json_hash(snapshot["object_tree"]),
        "info_sha256": stable_json_hash(snapshot["info"]),
        "done": snapshot["done"],
        "reward": snapshot["reward"],
    }

baseline_snapshot = collect_observation(
    env, observation=initial_observation, info=initial_info
)
baseline_summary = compact_state_summary(baseline_snapshot)
print(json.dumps(baseline_summary, indent=2))


{
  "move_count": 0,
  "observation_sha256": "27a74b42deeaab662300a35d09e059b45fb6109206596ef17e4dabb1b385f4fb",
  "object_tree_sha256": "8ae3cf16c6896fc8b3fb4fbfd3a6e989529a214d64502d1b83c549b6b3244153",
  "info_sha256": "490b27149a389da300bcb39bdae6a85e73a6994f8cefef3f647de3d8e34082c1",
  "done": null,
  "reward": null
}


## 9. Test a zero-move observation action

The earlier exploration found that `look around` could be used as an observation action without consuming a simulator move in the tested setting. We re-test it here rather than assume it is universally free.

If it is not a valid action in the current state, the cell reports that and does not execute it.


In [9]:
observation_action = "look around"
valid_now = list(env.get_valid_action_object_combinations() or [])
print("Action valid in current state:", observation_action in valid_now)

before_moves = env.get_num_moves()
if observation_action in valid_now:
    obs_result = env.step(observation_action)
    if not isinstance(obs_result, tuple) or len(obs_result) != 4:
        raise RuntimeError(f"Unexpected step return structure: {type(obs_result)}")
    obs_text, obs_reward, obs_done, obs_info = obs_result
    after_moves = env.get_num_moves()
    print("Observation:", obs_text)
    print("Reward:", obs_reward, "| done:", obs_done)
    print("Move count before/after:", before_moves, after_moves)
    print("Move delta:", after_moves - before_moves)
else:
    print("Skipped: action is not currently valid.")


Action valid in current state: True
Observation: This room is called the hallway. In it, you see: 
	the agent
	a substance called air
	a picture
You also see:
	A door to the art studio (that is open)
	A door to the bedroom (that is open)
	A door to the greenhouse (that is open)
	A door to the kitchen (that is open)
	A door to the living room (that is open)
	A door to the workshop (that is open)
Reward: 0 | done: False
Move count before/after: 0 0
Move delta: 0


## 10. Test an invalid action safely

The purpose is to record how this installed environment handles an invalid action string. We do not assume that every invalid action is harmless or that invalid-action handling is identical across tasks.


In [10]:
invalid_action = "__invalid_exploration_action__"
before_moves = env.get_num_moves()

try:
    invalid_result = env.step(invalid_action)
    after_moves = env.get_num_moves()
    print("Returned value:", invalid_result)
    print("Move count before/after:", before_moves, after_moves)
    print("Move delta:", after_moves - before_moves)
except Exception as exc:
    after_moves = env.get_num_moves()
    print("Invalid action raised:", type(exc).__name__, str(exc))
    print("Move count before/after:", before_moves, after_moves)
    print("Move delta:", after_moves - before_moves)


Returned value: ('No known action matches that input.', 0, False, {'moves': 0, 'score': 0, 'reward': 0, 'look': 'This room is called the hallway. In it, you see: \n\tthe agent\n\ta substance called air\n\ta picture\nYou also see:\n\tA door to the art studio (that is open)\n\tA door to the bedroom (that is open)\n\tA door to the greenhouse (that is open)\n\tA door to the kitchen (that is open)\n\tA door to the living room (that is open)\n\tA door to the workshop (that is open)\n', 'inv': 'In your inventory, you see:\n\tan orange\n', 'taskDesc': 'Task Description:\nYour task is to boil water. For compounds without a boiling point, combusting the substance is also acceptable. First, focus on the substance. Then, take actions that will cause it to change its state of matter.', 'valid': ['close art studio door', 'close bedroom door', 'close door to greenhouse', 'close door to kitchen', 'close door to living room', 'close door to workshop', 'eat orange', 'focus on agent', 'focus on air', 'fo

## 11. Verify deterministic reset and action replay

A clean reset and replay are compared using observable outputs. A matching hash is evidence of repeatability for the captured channels, not proof that every hidden simulator variable is identical.


In [11]:
def make_fresh_env(task_name=TASK_NAME, variation_idx=VARIATION_INDEX, simplification=SIMPLIFICATION):
    test_env = ScienceWorldEnv(
        taskName=task_name, serverPath=None, envStepLimit=ENV_STEP_LIMIT
    )
    test_env.load(
        taskName=task_name,
        variationIdx=variation_idx,
        simplificationStr=simplification
    )
    obs, info = test_env.reset()
    return test_env, obs, info

replay_actions = ["look around"]

def replay_sequence():
    test_env, obs, info = make_fresh_env()
    trace = [collect_observation(test_env, obs, info, action="<RESET>")]
    for action in replay_actions:
        valid = list(test_env.get_valid_action_object_combinations() or [])
        if action not in valid:
            trace.append({"action": action, "skipped": "not_valid"})
            break
        obs, reward, done, info = test_env.step(action)
        trace.append(collect_observation(
            test_env, obs, info, reward, done, action
        ))
        if done:
            break
    test_env.close()
    return trace

trace_a = replay_sequence()
trace_b = replay_sequence()

hashes_a = [stable_json_hash(x) for x in trace_a]
hashes_b = [stable_json_hash(x) for x in trace_b]

determinism_result = {
    "task": TASK_NAME,
    "variation": VARIATION_INDEX,
    "actions": replay_actions,
    "trace_a_hashes": hashes_a,
    "trace_b_hashes": hashes_b,
    "observable_replay_match": hashes_a == hashes_b,
}
print(json.dumps(determinism_result, indent=2))
(RESULT_DIR / "determinism_replay_check.json").write_text(
    json.dumps(determinism_result, indent=2), encoding="utf-8"
)


{
  "task": "boil",
  "variation": 0,
  "actions": [
    "look around"
  ],
  "trace_a_hashes": [
    "1818b8b4864c578418ac18f01808c34010f884cee707f87354605927cae0c9be",
    "8fe182c3e52f81fe8256b4174e640d8abccc1935a3ff6f6f1fbe69dc69fc0708"
  ],
  "trace_b_hashes": [
    "1818b8b4864c578418ac18f01808c34010f884cee707f87354605927cae0c9be",
    "8fe182c3e52f81fe8256b4174e640d8abccc1935a3ff6f6f1fbe69dc69fc0708"
  ],
  "observable_replay_match": true
}


451

## 12. Inspect action templates and task-state signals

This section records available action templates and the current info structure. It does not infer that a task is successful or failed from a single boolean field without checking reward and task context.


In [12]:
possible_actions_result = env.get_possible_action_object_combinations()
print("Possible action object combinations return type:", type(possible_actions_result).__name__)
print("Possible action object combinations preview:")
print(str(possible_actions_result)[:10000])

print("\nCurrent run history:")
print(json.dumps(json_safe(env.get_run_history()), indent=2, ensure_ascii=False)[:12000])


Possible action object combinations return type: tuple
Possible action object combinations preview:
([{'action': 'look in agent', 'template_id': 5, 'obj_ids': [17592], 'type_ids': [0]}, {'action': 'look in air', 'template_id': 5, 'obj_ids': [17514], 'type_ids': [144]}, {'action': 'look in art studio', 'template_id': 5, 'obj_ids': [17451], 'type_ids': [132]}, {'action': 'look in art studio door', 'template_id': 5, 'obj_ids': [17580], 'type_ids': [140]}, {'action': 'look in bedroom', 'template_id': 5, 'obj_ids': [17325], 'type_ids': [132]}, {'action': 'look in bedroom door', 'template_id': 5, 'obj_ids': [17568], 'type_ids': [140]}, {'action': 'look in door to greenhouse', 'template_id': 5, 'obj_ids': [17577], 'type_ids': [140]}, {'action': 'look in door to kitchen', 'template_id': 5, 'obj_ids': [17562], 'type_ids': [140]}, {'action': 'look in door to living room', 'template_id': 5, 'obj_ids': [17565], 'type_ids': [140]}, {'action': 'look in door to workshop', 'template_id': 5, 'obj_ids':

## 13. Small controlled action experiment: kitchen picture

This is a limited exploratory test of whether an object can be moved and then returned to its original location. It is not a final reversibility label.

The cell reconstructs the task from a fresh reset, checks action validity before each action, and records each transition. If any action is unavailable, the sequence stops and records why.


In [13]:
def run_checked_sequence(task_name, variation_idx, actions, simplification=SIMPLIFICATION):
    test_env, obs, info = make_fresh_env(task_name, variation_idx, simplification)
    trace = [collect_observation(test_env, obs, info, action="<RESET>")]
    stopped_reason = None

    for action in actions:
        valid = list(test_env.get_valid_action_object_combinations() or [])
        if action not in valid:
            stopped_reason = f"Action not valid in current state: {action}"
            trace.append({"action": action, "skipped": "not_valid"})
            break

        obs, reward, done, info = test_env.step(action)
        trace.append(collect_observation(
            test_env, obs, info, reward, done, action
        ))
        if done:
            stopped_reason = f"Environment ended after action: {action}"
            break

    result = {
        "task": task_name,
        "variation": variation_idx,
        "actions_requested": actions,
        "trace": trace,
        "stopped_reason": stopped_reason,
    }
    test_env.close()
    return result

picture_sequence = [
    "go to kitchen",
    "pick up picture",
    "go to art studio",
    "put down picture",
    "go to art studio",
    "pick up picture",
    "go to hallway",
    "put down picture",
]

picture_experiment = run_checked_sequence(
    TASK_NAME, VARIATION_INDEX, picture_sequence
)
(RESULT_DIR / "picture_move_exploration.json").write_text(
    json.dumps(picture_experiment, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Stopped reason:", picture_experiment["stopped_reason"])
for i, step in enumerate(picture_experiment["trace"]):
    print(i, step.get("action"), "moves=", step.get("move_count"), "done=", step.get("done"))


Stopped reason: Action not valid in current state: pick up picture
0 <RESET> moves= 0 done= None
1 go to kitchen moves= 1 done= False
2 pick up picture moves= None done= None


## 14. Compare selected observable channels before and after the picture sequence

This comparison is deliberately descriptive. It does not declare the action reversible merely because selected fields match, nor irreversible because a fingerprint differs.


In [14]:
picture_trace = picture_experiment["trace"]
if len(picture_trace) >= 2:
    first = picture_trace[0]
    last = picture_trace[-1]
    comparison = {
        "initial_move_count": first.get("move_count"),
        "final_move_count": last.get("move_count"),
        "observation_matches": first.get("observation") == last.get("observation"),
        "object_tree_matches": stable_json_hash(first.get("object_tree")) == stable_json_hash(last.get("object_tree")),
        "info_matches": stable_json_hash(first.get("info")) == stable_json_hash(last.get("info")),
        "initial_object_tree_hash": stable_json_hash(first.get("object_tree")),
        "final_object_tree_hash": stable_json_hash(last.get("object_tree")),
        "interpretation": (
            "Selected-observable comparison only; not a complete simulator-state equivalence test."
        ),
    }
else:
    comparison = {"status": "insufficient_trace", "trace_length": len(picture_trace)}

(RESULT_DIR / "picture_observable_comparison.json").write_text(
    json.dumps(comparison, indent=2), encoding="utf-8"
)
print(json.dumps(comparison, indent=2))


{
  "initial_move_count": 0,
  "final_move_count": null,
  "observation_matches": false,
  "object_tree_matches": false,
  "info_matches": false,
  "initial_object_tree_hash": "8ae3cf16c6896fc8b3fb4fbfd3a6e989529a214d64502d1b83c549b6b3244153",
  "final_object_tree_hash": "74234e98afe7498fb5daf1f36ac2d78acc339464f950703b8c019892f982b90b",
  "interpretation": "Selected-observable comparison only; not a complete simulator-state equivalence test."
}


## 15. Inspect task completion semantics

This cell is a diagnostic reminder and a runtime check of the `done` and reward fields observed during the experiment. A terminal state may reflect success, failure, or another task-ending condition; inspect the accompanying reward and info.


In [15]:
terminal_events = []
for step in picture_experiment["trace"]:
    if step.get("done") is True:
        terminal_events.append({
            "action": step.get("action"),
            "reward": step.get("reward"),
            "info": step.get("info"),
            "move_count": step.get("move_count"),
        })

print("Terminal events observed:", len(terminal_events))
print(json.dumps(terminal_events, indent=2, ensure_ascii=False))
print("\nInterpretation rule: done=True alone is not an irreversibility label.")


Terminal events observed: 0
[]

Interpretation rule: done=True alone is not an irreversibility label.


## 16. Save the exploration report and machine-readable summary

The report records what was actually observed in this run. It does not carry forward old experimental findings.


In [16]:
runtime_summary = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "java_version_output": java_text,
    "scienceworld_version": SCIENCEWORLD_VERSION,
    "scienceworld_location": scienceworld.__file__,
    "task": TASK_NAME,
    "variation": VARIATION_INDEX,
    "simplification": SIMPLIFICATION,
    "available_task_count": len(task_names),
    "available_tasks": task_names,
    "initial_valid_action_count": len(valid_actions_initial),
    "observable_replay_match": determinism_result["observable_replay_match"],
    "picture_sequence_stopped_reason": picture_experiment["stopped_reason"],
    "terminal_events_in_picture_sequence": terminal_events,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

summary_path = RESULT_DIR / "nb1_exploration_summary.json"
summary_path.write_text(
    json.dumps(runtime_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

report_lines = [
    "# NB1 ScienceWorld Exploration Report",
    "",
    f"- ScienceWorld version: {SCIENCEWORLD_VERSION}",
    f"- Task: {TASK_NAME}",
    f"- Variation: {VARIATION_INDEX}",
    f"- Simplification: {SIMPLIFICATION}",
    f"- Available task count: {len(task_names)}",
    f"- Initial valid action count: {len(valid_actions_initial)}",
    f"- Observable replay matched: {determinism_result['observable_replay_match']}",
    f"- Picture sequence stopped reason: {picture_experiment['stopped_reason']}",
    "",
    "## Interpretation safeguards",
    "- This notebook is exploratory and assigns no definitive reversibility labels.",
    "- Matching selected observables is not proof of complete simulator-state restoration.",
    "- A changed object-tree fingerprint is not, by itself, proof of irreversibility.",
    "- done=True is a task outcome, not proof of irreversibility.",
    "- Failed recovery within this limited action sequence is not proof of global irreversibility.",
    "",
    "## Runtime summary",
    "```json",
    json.dumps(runtime_summary, indent=2, ensure_ascii=False),
    "```",
]
report_path = REPORT_DIR / "NB1_SCIENCEWORLD_EXPLORATION_REPORT.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

print("Saved:")
print(summary_path)
print(report_path)


Saved:
/content/irrev/results/scienceworld_exploration/nb1_exploration_summary.json
/content/irrev/reports/NB1_SCIENCEWORLD_EXPLORATION_REPORT.md


## 17. Validate the generated files

Check that expected outputs exist and that JSON files can be parsed. This is a lightweight integrity check, not a scientific validation of reversibility conclusions.


In [17]:
expected_files = [
    RESULT_DIR / "task_inventory.json",
    RESULT_DIR / "initial_environment_observation.json",
    RESULT_DIR / "initial_valid_actions.json",
    RESULT_DIR / "determinism_replay_check.json",
    RESULT_DIR / "picture_move_exploration.json",
    RESULT_DIR / "picture_observable_comparison.json",
    RESULT_DIR / "nb1_exploration_summary.json",
    REPORT_DIR / "NB1_SCIENCEWORLD_EXPLORATION_REPORT.md",
]

validation_rows = []
for path in expected_files:
    exists = path.exists()
    parse_ok = None
    if exists and path.suffix == ".json":
        try:
            json.loads(path.read_text(encoding="utf-8"))
            parse_ok = True
        except Exception:
            parse_ok = False
    validation_rows.append({
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": exists,
        "json_parse_ok": parse_ok,
        "bytes": path.stat().st_size if exists else None,
    })

print(json.dumps(validation_rows, indent=2))
if not all(row["exists"] for row in validation_rows):
    raise RuntimeError("One or more expected NB1 outputs are missing.")
if not all(row["json_parse_ok"] is not False for row in validation_rows):
    raise RuntimeError("One or more JSON outputs failed to parse.")


[
  {
    "path": "results/scienceworld_exploration/task_inventory.json",
    "exists": true,
    "json_parse_ok": true,
    "bytes": 1034
  },
  {
    "path": "results/scienceworld_exploration/initial_environment_observation.json",
    "exists": true,
    "json_parse_ok": true,
    "bytes": 300739
  },
  {
    "path": "results/scienceworld_exploration/initial_valid_actions.json",
    "exists": true,
    "json_parse_ok": true,
    "bytes": 2780
  },
  {
    "path": "results/scienceworld_exploration/determinism_replay_check.json",
    "exists": true,
    "json_parse_ok": true,
    "bytes": 451
  },
  {
    "path": "results/scienceworld_exploration/picture_move_exploration.json",
    "exists": true,
    "json_parse_ok": true,
    "bytes": 724298
  },
  {
    "path": "results/scienceworld_exploration/picture_observable_comparison.json",
    "exists": true,
    "json_parse_ok": true,
    "bytes": 449
  },
  {
    "path": "results/scienceworld_exploration/nb1_exploration_summary.json",
    

## 18. Create verified timestamped backups to both Drive roots

Each run creates a new timestamped snapshot in both Drive locations. Existing snapshots are not overwritten. The manifest records relative paths, file sizes, and SHA-256 hashes; copied files are re-hashed and verified.


In [18]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
snapshot_name = f"NB1_snapshot_{timestamp}"

# Snapshot only this project's files; omit transient Python caches.
source_files = [
    p for p in PROJECT_ROOT.rglob("*")
    if p.is_file() and "__pycache__" not in p.parts
]

manifest_entries = []
for src in sorted(source_files):
    rel = src.relative_to(PROJECT_ROOT)
    manifest_entries.append({
        "relative_path": str(rel),
        "size_bytes": src.stat().st_size,
        "sha256": sha256_file(src),
    })

manifest = {
    "snapshot_name": snapshot_name,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_root": str(PROJECT_ROOT),
    "files": manifest_entries,
}

# Write manifest locally before copying so it is included in both snapshots.
manifest_path = PROJECT_ROOT / "reports" / f"{snapshot_name}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# Rebuild source list to include the manifest.
source_files = [
    p for p in PROJECT_ROOT.rglob("*")
    if p.is_file() and "__pycache__" not in p.parts
]

backup_results = []
for root in (DRIVE_ROOT, BACKUP_ROOT):
    destination = root / snapshot_name
    if destination.exists():
        raise FileExistsError(f"Snapshot already exists: {destination}")
    destination.mkdir(parents=True)

    copied = 0
    failures = []
    for src in source_files:
        rel = src.relative_to(PROJECT_ROOT)
        dst = destination / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        if sha256_file(src) != sha256_file(dst):
            failures.append(str(rel))
        else:
            copied += 1

    backup_results.append({
        "destination": str(destination),
        "files_verified": copied,
        "verification_failures": failures,
    })

print(json.dumps(backup_results, indent=2))
if any(item["verification_failures"] for item in backup_results):
    raise RuntimeError("Backup verification failed.")


[
  {
    "destination": "/content/drive/MyDrive/ir/NB1_snapshot_20261123T123037Z",
    "files_verified": 10,
    "verification_failures": []
  },
  {
    "destination": "/content/drive/MyDrive/ir_final/NB1_snapshot_20261123T123037Z",
    "files_verified": 10,
    "verification_failures": []
  }
]


## 19. Final NB1 status

Review the printed validation and backup results above before marking this notebook complete.

**Completion criteria**

- [ ] ScienceWorld 1.2.3 installed and imported.
- [ ] Java runtime verified.
- [ ] Task inventory saved.
- [ ] Environment reset and initial state observations saved.
- [ ] Valid action catalogue saved.
- [ ] Deterministic replay check completed and recorded.
- [ ] Exploratory action sequence and limitations documented.
- [ ] JSON outputs validated.
- [ ] Both Drive snapshots copied and SHA-256 verified.

**Important:** NB1 establishes environment familiarity and records exploratory observations. It does not produce the final reversibility methodology or dataset. Those belong to NB2 and NB3.
